# 3. External data

Reads files that never went through a questionnaire - published indicator
tables from another source, one folder per chapter under
`DATA COLLECTOR\external data\<Chapter>\` - reshapes them, and writes each
chapter's rows to `<Chapter>_EN_external.xlsx`, already in English.

```
DATA COLLECTOR\external data\<Chapter>\*.xlsx
        -> DATA COLLECTOR\external data\<Chapter>\*_reshaped.xlsx   (the questionnaire-layout copy, beside the source)
        -> merged_long_files\<Chapter>_EN_external.xlsx               (read by notebook 4)
```

**Runs before notebook 4, between it and notebook 2** - this notebook does
none of the translation itself. `<Chapter>_EN_external.xlsx` is an INPUT to
notebook 4 the same shape as `<Chapter>_EN_questionnaires.xlsx` is, except
notebook 4 also has to give it an Arabic form, since (unlike a real English
questionnaire) nothing else in this project has ever written this content
in Arabic. Splitting the work this way keeps every translation and
dictionary-gap decision in the one notebook that already owns them, rather
than this notebook keeping its own separate copy of that machinery.

## Reshaping into the questionnaire layout

Follows the `reshape-external-data-questionnaire-layout` skill: external
files are usually the opposite shape of a real questionnaire sheet - long
country/year panels with metrics spread across many columns, sometimes under
merged group headers - where a questionnaire sheet is
`Index | Indicator | Country | Breakdown | year-columns`, one sheet per
indicator. `save_questionnaire_layout()` below writes one reshaped workbook
per source file, beside the source file, styled to match that layout -
title row, bold bordered header, frozen panes - built from the SAME rows
`extract_sheet()` already produced for `<Chapter>_EN_external.xlsx`, not a
second reading of the sheet, so the deliverable can never disagree with
what actually gets written.

## Why interactive, not a script

**Structure inference cannot be done safely without a person looking at
it.** Nothing here assumes "row 1 is headers." The one real file used to
build this had six sheets and three different header shapes - single row, a
group row above it, a group row built into the anchor row itself with the
split one row below - discovered by reading the actual file, not by
guessing at a spec. A column with data but no header it could confidently
name is left out and reported rather than guessed at - but a column left
out no longer takes every OTHER column in the sheet down with it. It used
to: `4.10` in the sample file has one stray value with no header above it
anywhere, entered one column too far right in the source file, and refusing
the whole sheet over that one cell lost 781 otherwise-good figures across
all 22 countries. Two adjacent columns reading "Number of beds per 1000" /
"population" are a second, different kind of judgement call the same sheet
raises - see Known issues in `../CLAUDE.md` rather than guessing whether
that is one indicator wrapped across two cells or two genuinely separate
ones; this notebook takes the conservative reading and reports it rather
than inventing a merge.

## Marking what came from outside

Every row written gets `Data Origin` = `External` - blank for every row
that was already there. Notebook 5's row and breakdown columns are a fixed,
named list (`ROW_COLUMNS`, `BREAKDOWN_COLUMNS`); `Data Origin` is in
neither, so tabulations already ignore it without anything there having to
change. The reshaped questionnaire-layout copy is a human-readable side
deliverable only - nothing in the real pipeline reads it back.

## Running it

Run the cells through **Run** below. `<Chapter>_EN_external.xlsx` is
overwritten in full each run - this notebook owns that file outright, so
there is nothing of a previous run to strip first the way notebook 4 has to
for the Arabic side. Report: sheets read, columns or sheets it could not
confidently name (and why). No dictionary gaps are found or filled here -
notebook 4 finds and fills whatever this notebook's output needs.


In [ ]:
"""
CELL: Imports and logging setup.
"""
from collections import OrderedDict, defaultdict
import logging
import re
from pathlib import Path

import openpyxl
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("compendium")


## Config

In [ ]:
"""
CELL: Configuration - paths, the chapters this looks for, and the new
column that marks a row as coming from outside the questionnaires.
"""
DATA_COLLECTOR_PATH = Path.home() / "OneDrive - United Nations" / "Desktop" / "DSS" / "DATA COLLECTOR"
EXTERNAL_DATA_PATH = DATA_COLLECTOR_PATH / "external data"
COMPENDIUM_PATH = Path.home() / "OneDrive - United Nations" / "Desktop" / "DSS" / "COMPENDIUM-ARAB SOCIETY"
LONG_FILES_PATH = COMPENDIUM_PATH / "merged_long_files"

# Leave as None to check every chapter with an external-data subfolder, or
# restrict e.g. ["Health"].
CHAPTERS = None

# Marks a row as not from the questionnaires. Blank for every existing row -
# nothing already in a long file is touched or backfilled with this - and set
# only on rows this notebook itself writes. Not in notebook 5's ROW_COLUMNS
# or BREAKDOWN_COLUMNS, so tabulations never see it; nothing there has to
# change for that to stay true.
ORIGIN_COLUMN_EN = "Data Origin"
ORIGIN_EXTERNAL_EN = "External"

# The six domains - both here and as `external data\<name>\` subfolder names.
KNOWN_CHAPTERS = ["Health", "Population", "Education", "Labor", "Poverty", "Housing"]


def external_file_path(chapter):
    """Where this chapter's extracted-but-not-yet-translated rows land -
    an INPUT to notebook 4, the same idea as <Chapter>_EN_questionnaires.xlsx
    except notebook 4 also has to give this one an Arabic form."""
    return LONG_FILES_PATH / f"{chapter}_EN_external.xlsx"


# One file per chapter - pipeline_inconsistencies_<Chapter>.txt - plus
# pipeline_inconsistencies_general.txt for anything with no chapter of its
# own. See save_inconsistencies() below.


def chapter_log_path(chapter):
    """Where one chapter's own findings live - every notebook's sections
    side by side, but never mixed with another chapter's."""
    return COMPENDIUM_PATH / f"pipeline_inconsistencies_{chapter}.txt"


GENERAL_LOG_PATH = COMPENDIUM_PATH / "pipeline_inconsistencies_general.txt"


def _write_section(path, title, section, section_text):
    """Replace one named section in one file, in place, leaving every other
    section exactly as it was. The mechanic every chapter file and the
    general one share - split what's there into sections by name, replace
    this one, rebuild in section order so the file reads the same whatever
    order the notebooks last ran in."""
    header = [title, "=" * 78, ""]

    sections = {}
    if path.exists():
        existing = path.read_text(encoding="utf-8")
        parts = re.split(r"^### (.+?) ###$", existing, flags=re.M)
        for name, text in zip(parts[1::2], parts[2::2]):
            sections[name] = f"### {name} ###{text.rstrip()}"
    sections[section] = section_text

    body_text = "\n\n".join(sections[name] for name in sorted(sections))
    path.write_text("\n".join(header).rstrip("\n") + "\n\n" + body_text + "\n",
                    encoding="utf-8")


def save_inconsistencies(section, records, chapters=None):
    """Write this notebook's findings for this section, one file per chapter
    - `pipeline_inconsistencies_<Chapter>.txt`, beside the codes folder - so
    a chapter-scoped run of one notebook never overwrites a different
    chapter's history the way one shared file used to. A finding with no
    chapter (a dictionary-level problem, not a source-data one) goes to
    `pipeline_inconsistencies_general.txt` instead.

    `chapters` is every chapter this call actually covers, independent of
    whether any of them have a finding - pass it explicitly so a clean
    chapter still gets its section correctly replaced with "Nothing found"
    rather than left showing whatever an earlier, unrelated run left there.
    """
    stamp = pd.Timestamp.now().strftime("%d %B %Y, %H:%M")
    marker = f"### {section} ###"

    WHERE = ["country", "indicator", "year", "sex", "age_group",
             "nationality", "area", "file", "sheet", "row"]

    def render(chapter_records):
        body = [marker, f"    last run {stamp}", ""]
        if not chapter_records:
            body += ["    Nothing found.", ""]
        else:
            frame = pd.DataFrame(chapter_records)
            for kind, group in frame.groupby("kind", sort=False):
                body.append(f"  {kind.upper()}  ({len(group)})")
                for _, row in group.iterrows():
                    def show(value):
                        if isinstance(value, float) and float(value).is_integer():
                            return str(int(value))
                        return str(value)
                    where = " · ".join(
                        show(row[f]) for f in WHERE
                        if f in row and pd.notna(row[f]) and str(row[f]) != "")
                    body.append(f"      {where}" if where else "      -")
                    body.append(f"          {row['detail']}")
                body.append("")
        return "\n".join(body)

    by_chapter = defaultdict(list)
    general = []
    for record in records:
        chapter = record.get("chapter")
        if chapter in (None, "", "-"):
            general.append(record)
        else:
            by_chapter[str(chapter)].append(record)

    covered = {str(c) for c in (chapters or [])} | set(by_chapter)

    paths = []
    for chapter in sorted(covered):
        path = chapter_log_path(chapter)
        _write_section(path, f"PIPELINE INCONSISTENCIES — {chapter}", section,
                       render(by_chapter.get(chapter, [])))
        paths.append(path)

    if general or not covered:
        _write_section(GENERAL_LOG_PATH, "PIPELINE INCONSISTENCIES — general", section,
                       render(general))
        paths.append(GENERAL_LOG_PATH)

    return paths, len(records)


def external_chapters():
    """Chapters with a folder under external data\\ holding at least one .xlsx."""
    if not EXTERNAL_DATA_PATH.exists():
        logger.warning(f"{EXTERNAL_DATA_PATH} does not exist - nothing to read")
        return []
    found = []
    names = CHAPTERS if CHAPTERS else KNOWN_CHAPTERS
    for name in names:
        folder = EXTERNAL_DATA_PATH / name
        if folder.exists() and any(f for f in folder.glob("*.xlsx") if not f.name.startswith("~$")):
            found.append(name)
    return found


## Structure inference - title, header shape, forward-fill

In [ ]:
"""
CELL: infer_sheet() - work out one raw sheet's shape and turn it into long rows.

Nothing here is a fixed "row 1 is headers" assumption - every real sheet in
the first external file broke that assumption in a different way. What is
reliable, across every shape seen so far, is that the row holding "Country"
then "Year" in two adjacent cells anchors everything else - whether or not
there is a group-label row, whichever side of the anchor it sits on, and
whichever column the pair happens to start at. A later file added its own
leading "index" counter column, shifting Country/Year one column right of
where the first file always had them - found by column NAME here, not a
fixed position, for the same reason notebook 6's breakdown_column() finds a
column by name: a position pinned to the one file seen so far survives
exactly until the next one doesn't share it.

  - group row ABOVE the anchor, anchor row itself holds the leaf labels
    (e.g. a "Population per one personnel" / "Health Personnel per 100,000"
    row above "Country, Year, Number of physicians, Number of dentists, ...");
  - group row ON the anchor row, leaf labels BELOW it
    (e.g. "Country, Year, Visual, Hearing, ..." then a "Male, Female, Total"
    row underneath each group);
  - no group at all - the anchor row's own cells are the leaf labels.

A merged group cell (openpyxl only keeps a value in the top-left cell of a
merge) is forward-filled rightward before use.

Confidence is refused, not guessed, when no row has "Country" then "Year" in
two adjacent cells at all, or when a column has real data below it but no
label a plan could be built for. A skip is logged with the reason, never
silently dropped.
"""

SEX_LABELS = {"male": "Male", "female": "Female", "total": "Both sexes"}

# A metric column literally headed "value"/"Values" carries no name of its
# own - the real name lives elsewhere on the sheet (a title row, or a
# per-row Indicator column - see indicator_column_title() below) - and is
# resolved the same way a Male/Female/Total column's name is: through the
# sheet's title, not the header text.
GENERIC_VALUE_LABELS = {"value", "values"}


def is_blank(value):
    return value is None or (isinstance(value, str) and value.strip() == "")


def clean_text(value):
    return "" if is_blank(value) else str(value).strip()


def collapse_whitespace(text):
    """A title or indicator name wrapped across lines in its source cell,
    read back as one cell with an embedded newline - flattened to one
    readable line rather than carried into every row it names."""
    return re.sub(r"\s+", " ", text).strip()


def find_header_anchor(ws, max_scan_row=15):
    """(row, country_column) where two adjacent cells read 'Country' then
    'Year', case-insensitively - or None if no such pair exists in the
    first max_scan_row rows. Country's column is not assumed to be column
    A: a leading column of its own (an "index" counter, say) shifts the
    pair right, and the pair is found by name rather than position so that
    shift costs nothing here.
    """
    for row in range(1, max_scan_row + 1):
        for column in range(1, ws.max_column):
            a = clean_text(ws.cell(row=row, column=column).value).lower()
            b = clean_text(ws.cell(row=row, column=column + 1).value).lower()
            if a == "country" and b == "year":
                return row, column
    return None


def read_row(ws, row, country_col, last_col):
    """Row values as a 0-indexed list starting at country_col, position i =
    column country_col+i. Position 0 is always Country, position 1 is
    always Year, for a row read at the anchor."""
    if row < 1:
        return [None] * (last_col - country_col + 1)
    return [ws.cell(row=row, column=c).value for c in range(country_col, last_col + 1)]


def forward_fill_right(values):
    """Blank cells inherit the last non-blank value to their left - how a
    merged header cell reads back once openpyxl un-merges it (only the
    top-left cell of a merge keeps its value; the rest come back as None).
    """
    filled, last = [], None
    for value in values:
        if not is_blank(value):
            last = value
        filled.append(last)
    return filled


def looks_like_label_row(values):
    """True only for a row that could be a header/group-label row: blank in
    the Country/Year positions - the way a real data row never is - with
    real content further along. Without the blank-Country/Year check, the
    very first DATA row of a plain, single-header-row sheet (no group at
    all) reads as "has content" too, and gets mistaken for a leaf-label row
    sitting below the header - which would read that row's own data values
    as if they were column labels.
    """
    return is_blank(values[0]) and is_blank(values[1]) and any(not is_blank(v) for v in values[2:])


def build_column_plan(ws, anchor_row, country_col, last_col):
    """What each column beyond Country/Year actually is, whichever of the
    three shapes described above this sheet turns out to be.

    Returns (plan, data_start_row) - `plan` is a list aligned to the metric
    columns (everything after Country/Year), each entry
    {"indicator": str or None, "sex": str or None, "use_title": bool} or
    None where a column carries nothing.
    """
    width = last_col - country_col + 1
    anchor_values = read_row(ws, anchor_row, country_col, last_col)
    above_values = read_row(ws, anchor_row - 1, country_col, last_col)
    below_values = read_row(ws, anchor_row + 1, country_col, last_col)

    above_has_content = looks_like_label_row(above_values)
    below_has_content = looks_like_label_row(below_values)

    if above_has_content:
        group_values = forward_fill_right(above_values)
        leaf_values = anchor_values
        data_start_row = anchor_row + 1
    elif below_has_content:
        group_values = forward_fill_right(anchor_values)
        leaf_values = below_values
        data_start_row = anchor_row + 2
    else:
        group_values = [None] * width
        leaf_values = anchor_values
        data_start_row = anchor_row + 1

    # A blank buffer row (or more) between the header block and the data is
    # common (4.7 has two). Skip forward past any fully-empty rows.
    max_row = ws.max_row
    while data_start_row <= max_row and all(
        is_blank(v) for v in read_row(ws, data_start_row, country_col, last_col)
    ):
        data_start_row += 1

    plan = []
    for position in range(2, width):  # the metric columns, relative to country_col
        sub_label = clean_text(leaf_values[position])
        group_label = clean_text(group_values[position])
        if sub_label == "" and group_label == "":
            plan.append(None)
            continue

        sub_key = sub_label.lower()
        if sub_key in SEX_LABELS:
            # The title always supplies the base name here, whether or not
            # there is a group - a bare "Visual" would become its own
            # tabulation sheet in notebook 5, one per Indicator, with no
            # "disability" anywhere in the name it appears under.
            plan.append({
                "indicator": group_label or None,
                "sex": SEX_LABELS[sub_key],
                "use_title": True,
            })
        elif sub_key in GENERIC_VALUE_LABELS:
            plan.append({"indicator": group_label or None, "sex": None, "use_title": True})
        else:
            name = f"{group_label} - {sub_label}" if group_label and sub_label else (sub_label or group_label)
            plan.append({"indicator": name, "sex": None, "use_title": False})

    return plan, data_start_row


## extract_sheet() - one sheet's plan, turned into rows

In [ ]:
"""
CELL: extract_sheet() - turn one raw sheet into long rows, or refuse it.
"""


def read_title(ws, anchor_row):
    """The title above the header block, if there is one - row 1's own text,
    with a leading "Table 4.7" - style prefix dropped since the table number
    means nothing outside this workbook."""
    if anchor_row is None or anchor_row <= 1:
        return None
    text = clean_text(ws.cell(row=1, column=1).value)
    if not text:
        return None
    return collapse_whitespace(re.sub(r"^Table\s+\S+\s*", "", text, flags=re.I).strip() or text)


def indicator_column_title(ws, anchor_row, country_col, data_start_row, last_row):
    """A per-sheet title read from an 'Indicator' data column, when the
    sheet carries one, immediately to the left of Country - a second file
    read here writes the indicator name once per row in its own column
    instead of once in a header. The text is the same on every row of a
    sheet built this way, so the first non-blank row's value stands for the
    whole sheet, the same role read_title()'s title row plays for a sheet
    that has one. Tried only when there is no title row, so a sheet that
    genuinely has both keeps the title row's wording.
    """
    indicator_col = country_col - 1
    if indicator_col < 1:
        return None
    header = clean_text(ws.cell(row=anchor_row, column=indicator_col).value).lower()
    if header != "indicator":
        return None
    for row in range(data_start_row, last_row + 1):
        value = clean_text(ws.cell(row=row, column=indicator_col).value)
        if value:
            return collapse_whitespace(value)
    return None


def extract_sheet(ws, sheet_name, file_name):
    """Returns (rows, note, flagged_columns, title). `rows` is a list of
    dicts ready to become a DataFrame, empty if this sheet was refused
    entirely; `note` explains what happened, for the run's report;
    `flagged_columns` is every column that had data but no header this
    could confidently name; `title` is whatever name this sheet resolved
    for itself (a title row, or a per-row Indicator column), for the
    questionnaire-layout copy to reuse rather than re-deriving it.

    A flagged column is left OUT of `rows` and reported by column letter and
    a sample value - it no longer takes every other column in the sheet down
    with it. Refusing the WHOLE sheet is reserved for when no header/date row
    exists at all, or literally no column could be named.
    """
    anchor = find_header_anchor(ws)
    if anchor is None:
        return [], "no 'Country'/'Year' header found in the first 15 rows - skipped", [], None
    anchor_row, country_col = anchor

    last_col = ws.max_column
    plan, data_start_row = build_column_plan(ws, anchor_row, country_col, last_col)
    title = (read_title(ws, anchor_row)
            or indicator_column_title(ws, anchor_row, country_col, data_start_row, ws.max_row))

    if all(p is None for p in plan):
        return [], "header row found but no column could be named - skipped", [], title
    if any(p and p["use_title"] and not title for p in plan):
        return [], "a Male/Female/Total column with no group and no title to name it by - skipped", [], title

    width = last_col - country_col + 1
    year_col = country_col + 1

    # A column with real data below it but no plan entry means the shape was
    # not understood for THAT column, not that it is empty - report it by
    # column letter and a sample value rather than inventing a name for it,
    # and leave every other, correctly-named column's data in rows.
    flagged_columns = []
    for position in range(2, width):
        if plan[position - 2] is not None:
            continue
        column = country_col + position
        for row in range(data_start_row, ws.max_row + 1):
            value = ws.cell(row=row, column=column).value
            if not is_blank(value):
                flagged_columns.append({
                    "column": get_column_letter(column),
                    "sample_row": row,
                    "sample_value": value,
                })
                break

    rows = []
    current_country = None
    for row in range(data_start_row, ws.max_row + 1):
        raw_country = ws.cell(row=row, column=country_col).value
        if not is_blank(raw_country):
            current_country = clean_text(raw_country)
        raw_year = ws.cell(row=row, column=year_col).value
        year = pd.to_numeric(raw_year, errors="coerce")
        if pd.isna(year) or current_country is None:
            continue  # a blank trailer row, a footnote line, or data before any country appeared

        for position in range(2, width):
            entry = plan[position - 2]
            if entry is None:
                continue
            column = country_col + position
            value = ws.cell(row=row, column=column).value
            number = pd.to_numeric(value, errors="coerce")
            if pd.isna(number):
                continue  # blank cell for this indicator/year - not a finding, just unreported

            if entry["use_title"]:
                indicator = f"{title} - {entry['indicator']}" if entry["indicator"] else title
            else:
                indicator = entry["indicator"]
            rows.append({
                "Country": current_country, "Year": int(year), "Value": float(number),
                "Indicator": indicator, "Sex": entry["sex"],
                "Source": f"{file_name}, sheet {sheet_name}",
            })

    if not rows:
        return [], "header understood but no usable data rows found - skipped", flagged_columns, title

    note = f"{len(rows):,} row(s) extracted, {sum(1 for p in plan if p)} indicator column(s)"
    if flagged_columns:
        detail = "; ".join(f"{c['column']}{c['sample_row']}={c['sample_value']!r}"
                           for c in flagged_columns)
        note += f" - {len(flagged_columns)} column(s) left out, no header could name them ({detail})"
    return rows, note, flagged_columns, title


## Reshaping into the questionnaire layout

`build_questionnaire_records()` and `write_questionnaire_sheet()` below
follow the `reshape-external-data-questionnaire-layout` skill: they group
the rows `extract_sheet()` already produced into
`Index | Indicator | Country | Breakdown | year-columns` records - `Index`
assigned once per distinct indicator, in the order it was first seen, the
same convention the reference questionnaire files use - and write one styled
sheet per source table. `save_questionnaire_layout()` writes one such
workbook per source file, beside the source file.

In [ ]:
"""
CELL: Reshaping one file's already-extracted rows into the questionnaire
layout, and saving that as its own workbook beside the source file.
"""

_HEADER_FONT = Font(name="Calibri", size=11, bold=True)
_TITLE_FONT = Font(name="Calibri", size=12, bold=True)
_BODY_FONT = Font(name="Calibri", size=11)
_CENTER = Alignment(horizontal="center", vertical="center", wrap_text=True)
_LEFT = Alignment(horizontal="left", vertical="center")
_THIN = Side(style="thin", color="B7B7B7")
_BORDER = Border(left=_THIN, right=_THIN, top=_THIN, bottom=_THIN)
_HEADER_FILL = PatternFill("solid", fgColor="D9E1F2")


def build_questionnaire_records(rows):
    """Groups one sheet's already-extracted rows into the skill's record
    shape - one record per (Indicator, Sex, Country), a year->value map.
    Built from the SAME rows appended to the long file, never a second
    reading of the sheet, so this can never disagree with what was appended.
    """
    index_by_indicator = {}
    grouped = OrderedDict()
    for row in rows:
        indicator = row["Indicator"]
        if indicator not in index_by_indicator:
            index_by_indicator[indicator] = len(index_by_indicator) + 1
        key = (indicator, row["Sex"], row["Country"])
        grouped.setdefault(key, {})[row["Year"]] = row["Value"]

    records = []
    for (indicator, sex, country), values in grouped.items():
        records.append({
            "index": index_by_indicator[indicator], "indicator": indicator,
            "country": country, "breakdown": sex, "values": values,
        })
    years = sorted({row["Year"] for row in rows})
    return records, years


def write_questionnaire_sheet(out_wb, sheet_name, title, records, breakdown_label, years):
    """One sheet in the questionnaire layout: title row, blank row, bold
    bordered header row, one data row per record, frozen panes below the
    header. Styling only - see build_questionnaire_records() for the data.
    """
    ws = out_wb.create_sheet(sheet_name[:31])

    has_breakdown = breakdown_label is not None and any(r.get("breakdown") for r in records)
    fixed_cols = ["Index", "Indicator", "Country"] + ([breakdown_label] if has_breakdown else [])
    n_fixed = len(fixed_cols)
    total_cols = n_fixed + len(years)

    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=max(total_cols, 1))
    tcell = ws.cell(row=1, column=1, value=title)
    tcell.font = _TITLE_FONT
    tcell.alignment = _LEFT

    header_row = 3
    for i, colname in enumerate(fixed_cols, start=1):
        c = ws.cell(row=header_row, column=i, value=colname)
        c.font, c.alignment, c.fill, c.border = _HEADER_FONT, _CENTER, _HEADER_FILL, _BORDER
    for j, yr in enumerate(years, start=n_fixed + 1):
        c = ws.cell(row=header_row, column=j, value=yr)
        c.font, c.alignment, c.fill, c.border = _HEADER_FONT, _CENTER, _HEADER_FILL, _BORDER

    r = header_row + 1
    for rec in sorted(records, key=lambda rc: (rc["index"], rc["country"], rc.get("breakdown") or "")):
        ic = ws.cell(row=r, column=1, value=rec["index"])
        ic.font, ic.alignment, ic.border = _BODY_FONT, _CENTER, _BORDER
        c2 = ws.cell(row=r, column=2, value=rec["indicator"])
        c2.font, c2.alignment, c2.border = _BODY_FONT, _LEFT, _BORDER
        c3 = ws.cell(row=r, column=3, value=rec["country"])
        c3.font, c3.alignment, c3.border = _BODY_FONT, _LEFT, _BORDER
        col_offset = n_fixed
        if has_breakdown:
            cb = ws.cell(row=r, column=4, value=rec.get("breakdown"))
            cb.font, cb.alignment, cb.border = _BODY_FONT, _LEFT, _BORDER
        for j, yr in enumerate(years, start=col_offset + 1):
            cell = ws.cell(row=r, column=j, value=rec["values"].get(yr))
            cell.font = _BODY_FONT
            cell.alignment = Alignment(horizontal="center")
            cell.border = _BORDER
        r += 1

    ws.column_dimensions["A"].width = 8
    ws.column_dimensions["B"].width = 55
    ws.column_dimensions["C"].width = 20
    start_idx = 4
    if has_breakdown:
        ws.column_dimensions["D"].width = 16
        start_idx = 5
    for j in range(start_idx, total_cols + 1):
        ws.column_dimensions[get_column_letter(j)].width = 10
    ws.freeze_panes = ws.cell(row=header_row + 1, column=n_fixed + 1).coordinate
    return ws


def save_questionnaire_layout(chapter, source_path, sheet_rows):
    """One workbook, one sheet per source sheet, saved beside the source
    file. `sheet_rows` is {sheet_name: (rows, title)} for every sheet this
    notebook actually extracted something from - a sheet that was refused
    entirely has no rows and is not in it."""
    if not sheet_rows:
        return None
    out_wb = openpyxl.Workbook()
    out_wb.remove(out_wb.active)
    for sheet_name, (rows, title) in sheet_rows.items():
        records, years = build_questionnaire_records(rows)
        has_sex = any(r["Sex"] for r in rows)
        write_questionnaire_sheet(out_wb, sheet_name, title or sheet_name, records,
                                  "Sex" if has_sex else None, years)
    out_path = source_path.with_name(f"{source_path.stem}_reshaped.xlsx")
    out_wb.save(out_path)
    logger.info(f"  {chapter}: questionnaire-layout reshape -> {out_path.name} "
               f"({len(sheet_rows)} sheet(s))")
    return out_path


## Reading a chapter's external-data folder

In [ ]:
"""
CELL: read_external_file() - every sheet in one workbook, extracted or
refused, plus the reshaped questionnaire-layout copy saved beside it.
"""

SKIPPED_SHEETS = []     # {chapter, file, sheet, detail} - every sheet this refused entirely
FLAGGED_COLUMNS = []    # {chapter, file, sheet, column, sample_row, sample_value} - columns left out


def read_external_file(path, chapter):
    """Every usable row from every sheet in one external-data workbook."""
    try:
        wb = openpyxl.load_workbook(path, data_only=True)
    except Exception as error:
        SKIPPED_SHEETS.append({
            "chapter": chapter, "file": path.name, "sheet": "-",
            "detail": f"could not open the file: {type(error).__name__}: {error}",
        })
        return []

    all_rows = []
    sheet_rows = {}
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        rows, note, flagged, title = extract_sheet(ws, sheet_name, path.name)
        logger.info(f"  {path.name} | {sheet_name}: {note}")
        if flagged:
            FLAGGED_COLUMNS.extend({
                "chapter": chapter, "file": path.name, "sheet": sheet_name, **column,
            } for column in flagged)
        if not rows:
            SKIPPED_SHEETS.append({
                "chapter": chapter, "file": path.name, "sheet": sheet_name, "detail": note,
            })
            continue
        all_rows.extend(rows)
        sheet_rows[sheet_name] = (rows, title)

    save_questionnaire_layout(chapter, path, sheet_rows)
    return all_rows


def read_external_chapter(chapter):
    """Every usable row from every .xlsx in one chapter's external-data folder."""
    folder = EXTERNAL_DATA_PATH / chapter
    files = sorted(f for f in folder.glob("*.xlsx")
                   if not f.name.startswith("~$") and not f.stem.endswith("_reshaped"))
    logger.info(f"{chapter}: {len(files)} external file(s)")

    rows = []
    for path in files:
        rows.extend(read_external_file(path, chapter))
    if not rows:
        return pd.DataFrame(columns=["Country", "Year", "Value", "Indicator", "Sex", "Source"])

    table = pd.DataFrame(rows)
    table["Chapter"] = chapter
    table[ORIGIN_COLUMN_EN] = ORIGIN_EXTERNAL_EN
    return table


## Run - part 1: read, append to English, find the gaps

In [ ]:
"""
CELL: Main run - reshape every chapter's external files, write each one's
questionnaire-layout copy, and save the extracted rows to
<Chapter>_EN_external.xlsx for notebook 4 to translate and append.
"""
SKIPPED_SHEETS.clear()
FLAGGED_COLUMNS.clear()

chapters = external_chapters()
print(f"Chapters with external data: {chapters}\n")

WRITTEN = {}   # chapter -> path written, for the summary below

for chapter in chapters:
    print(f"=== {chapter} ===")
    new_rows = read_external_chapter(chapter)
    if new_rows.empty:
        print(f"  nothing usable found\n")
        continue

    out_path = external_file_path(chapter)
    new_rows.to_excel(out_path, index=False, engine="openpyxl")
    WRITTEN[chapter] = out_path
    logger.info(f"  {chapter}: {len(new_rows):,} row(s) -> {out_path.name}")
    print()

print("=" * 70)
print(f"{sum(1 for _ in WRITTEN):,} chapter(s) written: "
     f"{', '.join(f'{c} ({p.name})' for c, p in WRITTEN.items()) or 'none'}")
print(f"{len(SKIPPED_SHEETS)} sheet(s) skipped entirely - see below")
print(f"{len(FLAGGED_COLUMNS)} column(s) left out of an otherwise-usable sheet - see below")

if SKIPPED_SHEETS:
    print("\nSkipped entirely:")
    for row in SKIPPED_SHEETS:
        print(f"  {row['chapter']} · {row['file']} · {row['sheet']}: {row['detail']}")

if FLAGGED_COLUMNS:
    print("\nColumns left out:")
    for row in FLAGGED_COLUMNS:
        print(f"  {row['chapter']} · {row['file']} · {row['sheet']} · "
             f"col {row['column']}: e.g. row {row['sample_row']} = {row['sample_value']!r}")

records = [{"kind": "sheet skipped", **row} for row in SKIPPED_SHEETS]
records += [{"kind": "column left out, no header could name it", "chapter": row["chapter"],
            "file": row["file"], "sheet": row["sheet"],
            "detail": f"column {row['column']}: e.g. row {row['sample_row']} = {row['sample_value']!r}"}
           for row in FLAGGED_COLUMNS]
paths, count = save_inconsistencies("3. EXTERNAL DATA", records, chapters=chapters)
print(f"\n{count} inconsistency(ies) recorded across {len(paths)} file(s)")
